# Lấy Dữ Liệu Cryptocurrency từ Bitget API

Notebook này hướng dẫn cách lấy dữ liệu cryptocurrency từ Bitget API một cách đơn giản và dễ hiểu.

## Tính năng chính:
- ✅ Lấy dữ liệu 1 coin hoặc nhiều coin cùng lúc
- ✅ Hệ thống cập nhật file thông minh (không tạo file mới mỗi lần)
- ✅ Phân tích thị trường cơ bản
- ✅ Loại bỏ dữ liệu trùng lặp tự động
- ✅ Giao diện sạch sẽ, không có emoji

In [39]:
# Import thư viện cần thiết
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

print("Đã import thành công!")

Đã import thành công!


In [40]:
# Cấu hình API
API_URL = "https://api.bitget.com/api/v2/spot/market/history-candles"
DELAY_TIME = 0.1  # Chờ 100ms giữa các request

print(f"API URL: {API_URL}")
print(f"Thời gian chờ: {DELAY_TIME}s")

API URL: https://api.bitget.com/api/v2/spot/market/history-candles
Thời gian chờ: 0.1s


In [41]:
def get_crypto_data(symbol, timeframe="1h", days=7):
    """
    Lấy dữ liệu cryptocurrency từ Bitget
    
    Args:
        symbol (str): Cặp giao dịch (VD: "BTCUSDT")
        timeframe (str): Khung thời gian ("1min", "5min", "1h", "1day")
        days (int): Số ngày lấy dữ liệu
    
    Returns:
        DataFrame: Dữ liệu giá cryptocurrency
    """
    
    # Tính timestamp kết thúc (hiện tại)
    end_time = int(time.time() * 1000)
    
    # Tính số lượng nến cần lấy dựa trên timeframe
    timeframe_minutes = {
        "1min": 1,
        "5min": 5, 
        "15min": 15,
        "1h": 60,
        "4h": 240,
        "1day": 1440
    }
    
    minutes_per_candle = timeframe_minutes.get(timeframe, 60)
    total_minutes = days * 24 * 60
    limit = min(200, total_minutes // minutes_per_candle)  # Tối đa 200 nến
    
    # Chuẩn bị parameters
    params = {
        'symbol': symbol,
        'granularity': timeframe,
        'endTime': str(end_time),
        'limit': str(limit)
    }
    
    try:
        print(f"Đang lấy dữ liệu {symbol} ({timeframe})...")
        
        # Gửi request
        response = requests.get(API_URL, params=params, timeout=10)
        
        if response.status_code != 200:
            print(f"Lỗi HTTP: {response.status_code}")
            return None
        
        data = response.json()
        
        if data.get('code') != '00000':
            print(f"Lỗi API: {data.get('msg')}")
            return None
        
        # Chuyển đổi thành DataFrame
        candles = data.get('data', [])
        
        if not candles:
            print("Không có dữ liệu")
            return None
        
        df = pd.DataFrame(candles, columns=[
            'timestamp', 'open', 'high', 'low', 'close', 
            'volume', 'quote_volume', 'usdt_volume'
        ])
        
        # Chuyển đổi kiểu dữ liệu
        df['timestamp'] = pd.to_numeric(df['timestamp'])
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
        
        # Chuyển đổi giá thành số
        price_cols = ['open', 'high', 'low', 'close', 'volume']
        for col in price_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Sắp xếp theo thời gian
        df = df.sort_values('timestamp').reset_index(drop=True)
        
        print(f"Lấy thành công {len(df)} nến từ {df['datetime'].min()} đến {df['datetime'].max()}")
        
        return df[['datetime', 'open', 'high', 'low', 'close', 'volume']]  # Chỉ trả về cột cần thiết
        
    except Exception as e:
        print(f"Lỗi: {e}")
        return None

In [42]:
# VÍ DỤ 1: Lấy dữ liệu Bitcoin 1 giờ trong 7 ngày
print("=== VÍ DỤ 1: BITCOIN (BTC) ===\n")

btc_data = get_crypto_data("BTCUSDT", "1h", 7)

if btc_data is not None:
    print(f"\nThông tin dữ liệu:")
    print(f"   - Số lượng: {len(btc_data)} nến")
    print(f"   - Giá hiện tại: ${btc_data['close'].iloc[-1]:,.2f}")
    print(f"   - Giá cao nhất: ${btc_data['high'].max():,.2f}")
    print(f"   - Giá thấp nhất: ${btc_data['low'].min():,.2f}")
    
    print("\n5 dòng đầu tiên:")
    display(btc_data.head())
else:
    print("Không thể lấy dữ liệu Bitcoin")

time.sleep(DELAY_TIME)

=== VÍ DỤ 1: BITCOIN (BTC) ===

Đang lấy dữ liệu BTCUSDT (1h)...
Lấy thành công 168 nến từ 2025-05-29 02:00:00 đến 2025-06-05 01:00:00

Thông tin dữ liệu:
   - Số lượng: 168 nến
   - Giá hiện tại: $104,796.73
   - Giá cao nhất: $108,868.02
   - Giá thấp nhất: $103,087.00

5 dòng đầu tiên:
Lấy thành công 168 nến từ 2025-05-29 02:00:00 đến 2025-06-05 01:00:00

Thông tin dữ liệu:
   - Số lượng: 168 nến
   - Giá hiện tại: $104,796.73
   - Giá cao nhất: $108,868.02
   - Giá thấp nhất: $103,087.00

5 dòng đầu tiên:


,datetime,open,high,low,close,volume
0,2025-05-29 02:00:00,108167.01,108488.38,107964.00,108430.09,274.540272
1,2025-05-29 03:00:00,108430.09,108444.86,107989.00,108047.37,240.585296
2,2025-05-29 04:00:00,108047.37,108128.32,106978.47,107578.91,377.724040
3,2025-05-29 05:00:00,107578.91,107800.01,107517.00,107684.02,194.626992
4,2025-05-29 06:00:00,107684.02,107944.00,107670.00,107925.91,133.558434


In [43]:
# VÍ DỤ 2: Lấy dữ liệu nhiều coin cùng lúc - Phiên bản nâng cao
print("=== VÍ DỤ 2: NHIỀU CRYPTOCURRENCY - PHIÊN BẢN NÂNG CAO ===\n")

def get_multiple_crypto_data(coin_list, timeframe="1day", days=30):
    """
    Lấy dữ liệu nhiều cryptocurrency cùng lúc
    
    Args:
        coin_list (list): Danh sách các coin (VD: ["BTCUSDT", "ETHUSDT"])
        timeframe (str): Khung thời gian
        days (int): Số ngày lấy dữ liệu
    
    Returns:
        dict: Dictionary chứa dữ liệu của từng coin
    """
    all_data = {}
    summary_data = []
    
    print(f"Đang lấy dữ liệu {len(coin_list)} coin...")
    print("-" * 60)
    
    for i, coin in enumerate(coin_list, 1):
        print(f"[{i}/{len(coin_list)}] Đang xử lý {coin}...", end=" ")
        
        data = get_crypto_data(coin, timeframe, days)
        
        if data is not None:
            all_data[coin] = data
            
            # Tính toán thống kê cho mỗi coin
            current_price = data['close'].iloc[-1]
            prev_price = data['close'].iloc[-2] if len(data) > 1 else current_price
            first_price = data['close'].iloc[0]
            
            # Thay đổi 24h (hoặc 1 period trước)
            change_24h = ((current_price - prev_price) / prev_price) * 100
            
            # Thay đổi từ đầu kỳ
            total_change = ((current_price - first_price) / first_price) * 100
            
            # Tính high/low trong kỳ
            period_high = data['high'].max()
            period_low = data['low'].min()
            
            # Khối lượng trung bình
            avg_volume = data['volume'].mean()
            
            summary_data.append({
                'Coin': coin.replace('USDT', ''),
                'Giá hiện tại': current_price,
                'Thay đổi 24h (%)': change_24h,
                f'Thay đổi {days}d (%)': total_change,
                'Cao nhất': period_high,
                'Thấp nhất': period_low,
                'Volume TB': avg_volume
            })
            
            print("Thành công")
        else:
            print("Lỗi")
        
        time.sleep(DELAY_TIME)
    
    # Tạo DataFrame tóm tắt
    summary_df = pd.DataFrame(summary_data)
    
    return all_data, summary_df

# Danh sách các coin phổ biến
popular_coins = [
    "BTCUSDT",    # Bitcoin
    "ETHUSDT",    # Ethereum  
    "BNBUSDT",    # Binance Coin
    "ADAUSDT",    # Cardano
    "SOLUSDT",    # Solana
    "XRPUSDT",    # Ripple
    "DOTUSDT",    # Polkadot
    "MATICUSDT",  # Polygon
    "LINKUSDT",   # Chainlink
    "AVAXUSDT"    # Avalanche
]

# Lấy dữ liệu nhiều coin
crypto_data, summary = get_multiple_crypto_data(popular_coins, "1day", 30)

print(f"\nHoàn thành! Lấy được dữ liệu {len(crypto_data)} coin")
print("\nBẢNG TỔNG QUAN THỊ TRƯỜNG:")
print("=" * 80)

# Hiển thị bảng tóm tắt đẹp
if len(summary) > 0:
    # Format số đẹp hơn
    summary_display = summary.copy()
    summary_display['Giá hiện tại'] = summary_display['Giá hiện tại'].apply(lambda x: f"${x:,.2f}")
    summary_display['Thay đổi 24h (%)'] = summary_display['Thay đổi 24h (%)'].apply(lambda x: f"{x:+.2f}%")
    summary_display[f'Thay đổi 30d (%)'] = summary_display[f'Thay đổi 30d (%)'].apply(lambda x: f"{x:+.2f}%")
    summary_display['Cao nhất'] = summary_display['Cao nhất'].apply(lambda x: f"${x:,.2f}")
    summary_display['Thấp nhất'] = summary_display['Thấp nhất'].apply(lambda x: f"${x:,.2f}")
    summary_display['Volume TB'] = summary_display['Volume TB'].apply(lambda x: f"{x:,.0f}")
    
    display(summary_display)

print("\nTOP PERFORMERS (24h):")
if len(summary) > 0:
    # Sắp xếp theo thay đổi 24h
    top_gainers = summary.nlargest(3, 'Thay đổi 24h (%)')
    top_losers = summary.nsmallest(3, 'Thay đổi 24h (%)')
    
    print("Tăng mạnh nhất:")
    for _, coin in top_gainers.iterrows():
        print(f"   {coin['Coin']}: {coin['Thay đổi 24h (%)']:+.2f}% (${coin['Giá hiện tại']:,.2f})")
    
    print("\nGiảm mạnh nhất:")
    for _, coin in top_losers.iterrows():
        print(f"   {coin['Coin']}: {coin['Thay đổi 24h (%)']:+.2f}% (${coin['Giá hiện tại']:,.2f})")

# Tính tổng thị trường (giả định)
if len(summary) > 0:
    total_value = summary['Giá hiện tại'].sum()
    avg_change_24h = summary['Thay đổi 24h (%)'].mean()
    avg_change_30d = summary[f'Thay đổi 30d (%)'].mean()
    
    print(f"\nTHỐNG KÊ TỔNG QUAN:")
    print(f"   Tổng giá trị: ${total_value:,.2f}")
    print(f"   TB thay đổi 24h: {avg_change_24h:+.2f}%")
    print(f"   TB thay đổi 30d: {avg_change_30d:+.2f}%")

=== VÍ DỤ 2: NHIỀU CRYPTOCURRENCY - PHIÊN BẢN NÂNG CAO ===

Đang lấy dữ liệu 10 coin...
------------------------------------------------------------
[1/10] Đang xử lý BTCUSDT... Đang lấy dữ liệu BTCUSDT (1day)...
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:00
Thành công
[2/10] Đang xử lý ETHUSDT... Đang lấy dữ liệu ETHUSDT (1day)...
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:00
Thành công
[2/10] Đang xử lý ETHUSDT... Đang lấy dữ liệu ETHUSDT (1day)...
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:00
Thành công
[3/10] Đang xử lý BNBUSDT... Đang lấy dữ liệu BNBUSDT (1day)...
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:00
Thành công
[3/10] Đang xử lý BNBUSDT... Đang lấy dữ liệu BNBUSDT (1day)...
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:00
Thành công
[4/10] Đang xử lý ADAUSDT... Đang lấy dữ liệu ADAUSDT (1day)...
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:0

,Coin,Giá hiện tại,Thay đổi 24h (%),Thay đổi 30d (%),Cao nhất,Thấp nhất,Volume TB
0,BTC,"$105,306.45",-1.19%,+11.37%,"$111,958.32","$93,399.12","8,474"
1,ETH,"$2,667.90",+1.68%,+50.70%,"$2,788.01","$1,751.94","216,570"
2,BNB,$669.20,+0.34%,+11.81%,$697.60,$593.20,"5,043"
3,ADA,$0.69,-0.69%,+5.53%,$0.86,$0.64,"32,746,244"
4,SOL,$157.11,-3.14%,+9.73%,$187.68,$141.45,"1,172,546"
5,XRP,$2.25,-0.62%,+6.57%,$2.65,$2.08,"43,119,966"
6,DOT,$4.11,-1.82%,+5.12%,$5.39,$3.82,"1,307,966"
7,LINK,$14.18,-1.05%,+5.98%,$17.97,$13.21,"739,263"
8,AVAX,$21.10,-1.26%,+7.32%,$26.82,$19.09,"450,141"



TOP PERFORMERS (24h):
Tăng mạnh nhất:
   ETH: +1.68% ($2,667.90)
   BNB: +0.34% ($669.20)
   XRP: -0.62% ($2.25)

Giảm mạnh nhất:
   SOL: -3.14% ($157.11)
   DOT: -1.82% ($4.11)
   AVAX: -1.26% ($21.10)

THỐNG KÊ TỔNG QUAN:
   Tổng giá trị: $108,842.98
   TB thay đổi 24h: -0.86%
   TB thay đổi 30d: +12.68%


In [44]:
# Lưu dữ liệu nhiều coin ra nhiều file CSV - CẬP NHẬT THÔNG MINH NÂNG CAO
print("\n=== LƯU DỮ LIỆU NHIỀU COIN - CẬP NHẬT THÔNG MINH NÂNG CAO ===\n")

def save_multiple_crypto_data_smart(crypto_data_dict, folder_name="crypto_data"):
    """
    Lưu dữ liệu nhiều coin với hệ thống cập nhật thông minh
    Tự động gộp dữ liệu mới với dữ liệu cũ, loại bỏ trùng lặp
    
    Args:
        crypto_data_dict (dict): Dictionary chứa dữ liệu các coin
        folder_name (str): Thư mục lưu file
    
    Returns:
        dict: Thông tin chi tiết về các file đã xử lý
    """
    import os
    
    # Tạo thư mục nếu chưa có
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f"Đã tạo thư mục: {folder_name}")
    
    processing_results = {
        'success': [],
        'failed': [],
        'updated': [],
        'created': []
    }
    
    print(f"Bắt đầu xử lý {len(crypto_data_dict)} coin...")
    print("-" * 70)
    
    for coin, data in crypto_data_dict.items():
        try:
            if data is None:
                print(f"Bỏ qua {coin}: Không có dữ liệu")
                processing_results['failed'].append(coin)
                continue
            
            coin_name = coin.replace('USDT', '').lower()
            filename = f"{folder_name}/{coin_name}_data.csv"
            
            print(f"Xử lý {coin_name.upper()}...", end=" ")
            
            # Chuẩn bị dữ liệu với thông tin bổ sung
            data_enhanced = data.copy()
            data_enhanced['symbol'] = coin
            data_enhanced['price_change_pct'] = data_enhanced['close'].pct_change() * 100
            data_enhanced['high_low_pct'] = ((data_enhanced['high'] - data_enhanced['low']) / data_enhanced['low']) * 100
            data_enhanced['volume_ma_5'] = data_enhanced['volume'].rolling(window=5).mean()
            data_enhanced['price_ma_5'] = data_enhanced['close'].rolling(window=5).mean()
            data_enhanced['price_ma_20'] = data_enhanced['close'].rolling(window=20).mean()
            data_enhanced['updated_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            # Xử lý file
            if os.path.exists(filename):
                # File đã tồn tại - cập nhật
                try:
                    existing_data = pd.read_csv(filename)
                    original_count = len(existing_data)
                    
                    # Gộp và làm sạch dữ liệu
                    combined_data = pd.concat([existing_data, data_enhanced], ignore_index=True)
                    
                    # Loại bỏ trùng lặp và sắp xếp
                    combined_data = combined_data.drop_duplicates(subset=['datetime'], keep='last')
                    combined_data = combined_data.sort_values('datetime').reset_index(drop=True)
                    
                    # Lưu dữ liệu
                    combined_data.to_csv(filename, index=False)
                    
                    new_count = len(combined_data)
                    added_records = len(data_enhanced)
                    
                    print(f"Cập nhật (+{added_records} → {new_count} total)")
                    
                    processing_results['updated'].append({
                        'coin': coin,
                        'file': filename,
                        'original_records': original_count,
                        'added_records': added_records,
                        'total_records': new_count
                    })
                    
                except Exception as e:
                    print(f"Lỗi cập nhật, tạo mới ({str(e)[:30]}...)")
                    data_enhanced.to_csv(filename, index=False)
                    processing_results['created'].append({
                        'coin': coin,
                        'file': filename,
                        'records': len(data_enhanced)
                    })
            else:
                # File mới
                data_enhanced.to_csv(filename, index=False)
                print(f"Tạo mới ({len(data_enhanced)} records)")
                
                processing_results['created'].append({
                    'coin': coin,
                    'file': filename,
                    'records': len(data_enhanced)
                })
            
            processing_results['success'].append(coin)
            
        except Exception as e:
            print(f"Lỗi xử lý {coin}: {e}")
            processing_results['failed'].append(coin)
    
    # In báo cáo tổng kết
    print("\n" + "=" * 70)
    print("BÁO CÁO XỬ LÝ:")
    print(f"   Thành công: {len(processing_results['success'])} coin")
    print(f"   Thất bại: {len(processing_results['failed'])} coin")
    print(f"   File cập nhật: {len(processing_results['updated'])}")
    print(f"   File tạo mới: {len(processing_results['created'])}")
    
    if processing_results['updated']:
        print("\nCHI TIẾT CẬP NHẬT:")
        for item in processing_results['updated']:
            coin_display = item['coin'].replace('USDT', '')
            print(f"   {coin_display}: {item['original_records']} + {item['added_records']} = {item['total_records']} dòng")
    
    if processing_results['created']:
        print("\nFILE TẠO MỚI:")
        for item in processing_results['created']:
            coin_display = item['coin'].replace('USDT', '')
            print(f"   {coin_display}: {item['records']} dòng")
    
    if processing_results['failed']:
        print(f"\nLỖI: {', '.join(processing_results['failed'])}")
    
    return processing_results

# Áp dụng hàm mới cho dữ liệu đã lấy
if 'crypto_data' in locals() and len(crypto_data) > 0:
    results = save_multiple_crypto_data_smart(crypto_data)
    
    print(f"\nHoàn thành xử lý! Kiểm tra thư mục 'crypto_data' để xem các file.")
else:
    print("Không có dữ liệu để lưu. Hãy chạy phần lấy dữ liệu nhiều coin trước.")


=== LƯU DỮ LIỆU NHIỀU COIN - CẬP NHẬT THÔNG MINH NÂNG CAO ===

Bắt đầu xử lý 9 coin...
----------------------------------------------------------------------
Xử lý BTC... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý ETH... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý BNB... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý ADA... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý SOL... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý XRP... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý DOT... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý LINK... Lỗi cập nhật, tạo mới ('<' not supported between inst...)
Xử lý AVAX... Lỗi cập nhật, tạo mới ('<' not supported between inst...)

BÁO CÁO XỬ LÝ:
   Thành công: 9 coin
   Thất bại: 0 coin
   File cập nhật: 0
   File tạo mới: 9

FILE TẠO MỚI:
   BTC: 30 dòng
   ETH: 30 dòng
   BNB: 30 dòng
   ADA: 30 dòng
   SOL: 30 dòng
   XRP: 

In [45]:
# VÍ DỤ 3: Lưu dữ liệu ra file CSV - CẬP NHẬT THÔNG MINH
print("=== VÍ DỤ 3: LƯU DỮ LIỆU - CẬP NHẬT THÔNG MINH ===\n")

def save_crypto_data_smart(data, symbol, folder_name="crypto_data"):
    """
    Lưu dữ liệu cryptocurrency với cập nhật thông minh
    Cập nhật file hiện có thay vì tạo file mới
    
    Args:
        data (DataFrame): Dữ liệu cryptocurrency
        symbol (str): Ký hiệu coin (VD: "BTCUSDT")
        folder_name (str): Thư mục lưu file
    
    Returns:
        str: Đường dẫn file đã lưu
    """
    import os
    
    if data is None:
        print("Không có dữ liệu để lưu")
        return None
    
    # Tạo thư mục nếu chưa có
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f"Đã tạo thư mục: {folder_name}")
    
    # Tạo tên file cố định (không có timestamp)
    coin_name = symbol.replace('USDT', '').lower()
    filename = f"{folder_name}/{coin_name}_data.csv"
    
    # Thêm thông tin bổ sung vào dữ liệu
    data_enhanced = data.copy()
    data_enhanced['symbol'] = symbol
    data_enhanced['price_change_pct'] = data_enhanced['close'].pct_change() * 100
    data_enhanced['volume_ma_5'] = data_enhanced['volume'].rolling(window=5).mean()
    data_enhanced['price_ma_5'] = data_enhanced['close'].rolling(window=5).mean()
    data_enhanced['updated_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Kiểm tra file có tồn tại không
    if os.path.exists(filename):
        try:
            # Đọc dữ liệu cũ
            existing_data = pd.read_csv(filename)
            print(f"Tìm thấy file existing: {filename} ({len(existing_data)} dòng)")
            
            # Gộp dữ liệu mới với dữ liệu cũ
            combined_data = pd.concat([existing_data, data_enhanced], ignore_index=True)
            
            # Loại bỏ trùng lặp dựa trên datetime, giữ lại dữ liệu mới nhất
            combined_data = combined_data.drop_duplicates(subset=['datetime'], keep='last')
            
            # Sắp xếp theo thời gian
            combined_data = combined_data.sort_values('datetime').reset_index(drop=True)
            
            # Lưu dữ liệu đã gộp
            combined_data.to_csv(filename, index=False)
            
            new_records = len(data_enhanced)
            total_records = len(combined_data)
            
            print(f"Đã cập nhật {coin_name.upper()}: {filename}")
            print(f"   - Thêm mới: {new_records} dòng")
            print(f"   - Tổng cộng: {total_records} dòng")
            print(f"   - Thời gian: {combined_data['datetime'].min()} đến {combined_data['datetime'].max()}")
            
        except Exception as e:
            print(f"Lỗi khi đọc file cũ: {e}")
            print("Tạo file mới...")
            data_enhanced.to_csv(filename, index=False)
            print(f"Đã tạo file mới: {filename} ({len(data_enhanced)} dòng)")
    else:
        # Tạo file mới
        data_enhanced.to_csv(filename, index=False)
        print(f"Đã tạo file mới {coin_name.upper()}: {filename} ({len(data_enhanced)} dòng)")
    
    return filename

# Test với dữ liệu Bitcoin
if 'btc_data' in locals() and btc_data is not None:
    saved_file = save_crypto_data_smart(btc_data, "BTCUSDT")
    
    if saved_file:
        print(f"\nFile đã lưu: {saved_file}")
        
        # Hiển thị thông tin file
        saved_data = pd.read_csv(saved_file)
        print(f"\nThông tin file:")
        print(f"   - Tổng số dòng: {len(saved_data)}")
        print(f"   - Các cột: {list(saved_data.columns)}")
        print(f"   - Dữ liệu từ: {saved_data['datetime'].min()}")
        print(f"   - Dữ liệu đến: {saved_data['datetime'].max()}")
else:
    print("Không có dữ liệu Bitcoin để lưu. Hãy chạy cell lấy dữ liệu trước.")

=== VÍ DỤ 3: LƯU DỮ LIỆU - CẬP NHẬT THÔNG MINH ===

Tìm thấy file existing: crypto_data/btc_data.csv (30 dòng)
Lỗi khi đọc file cũ: '<' not supported between instances of 'Timestamp' and 'str'
Tạo file mới...
Đã tạo file mới: crypto_data/btc_data.csv (168 dòng)

File đã lưu: crypto_data/btc_data.csv

Thông tin file:
   - Tổng số dòng: 168
   - Các cột: ['datetime', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'price_change_pct', 'volume_ma_5', 'price_ma_5', 'updated_at']
   - Dữ liệu từ: 2025-05-29 02:00:00
   - Dữ liệu đến: 2025-06-05 01:00:00


In [46]:
# VÍ DỤ 4: Hàm đơn giản để lấy giá hiện tại
def get_current_price(symbol):
    """
    Lấy giá hiện tại của một cryptocurrency
    
    Args:
        symbol (str): Cặp giao dịch (VD: "BTCUSDT")
    
    Returns:
        float: Giá hiện tại
    """
    data = get_crypto_data(symbol, "1min", 1)
    
    if data is not None and len(data) > 0:
        return data['close'].iloc[-1]
    return None

# Test hàm lấy giá
print("=== VÍ DỤ 4: GIÁ HIỆN TẠI ===\n")

test_coins = ["BTCUSDT", "ETHUSDT", "BNBUSDT"]

for coin in test_coins:
    price = get_current_price(coin)
    
    if price:
        coin_name = coin.replace('USDT', '')
        print(f"{coin_name}: ${price:,.2f}")
    else:
        print(f"Không thể lấy giá {coin}")
    
    time.sleep(DELAY_TIME)

=== VÍ DỤ 4: GIÁ HIỆN TẠI ===

Đang lấy dữ liệu BTCUSDT (1min)...
Lấy thành công 200 nến từ 2025-06-04 23:35:00 đến 2025-06-05 02:54:00
BTC: $105,047.99
Đang lấy dữ liệu ETHUSDT (1min)...
Lấy thành công 200 nến từ 2025-06-04 23:35:00 đến 2025-06-05 02:54:00
BTC: $105,047.99
Đang lấy dữ liệu ETHUSDT (1min)...
Lấy thành công 200 nến từ 2025-06-04 23:35:00 đến 2025-06-05 02:54:00
ETH: $2,620.54
Đang lấy dữ liệu BNBUSDT (1min)...
Lấy thành công 200 nến từ 2025-06-04 23:35:00 đến 2025-06-05 02:54:00
ETH: $2,620.54
Đang lấy dữ liệu BNBUSDT (1min)...
Lấy thành công 200 nến từ 2025-06-04 23:35:00 đến 2025-06-05 02:54:00
BNB: $665.20
Lấy thành công 200 nến từ 2025-06-04 23:35:00 đến 2025-06-05 02:54:00
BNB: $665.20


In [47]:
# BONUS: Hàm tính thống kê đơn giản
def analyze_crypto(symbol, timeframe="1day", days=30):
    """
    Phân tích đơn giản một cryptocurrency
    
    Args:
        symbol (str): Cặp giao dịch
        timeframe (str): Khung thời gian
        days (int): Số ngày phân tích
    """
    print(f"\nPHÂN TÍCH {symbol.replace('USDT', '')} ({days} ngày)")
    print("=" * 40)
    
    data = get_crypto_data(symbol, timeframe, days)
    
    if data is None:
        print("Không thể lấy dữ liệu")
        return
    
    # Tính toán thống kê
    current_price = data['close'].iloc[-1]
    highest_price = data['high'].max()
    lowest_price = data['low'].min()
    avg_price = data['close'].mean()
    avg_volume = data['volume'].mean()
    
    # Thay đổi giá từ đầu kỳ
    first_price = data['close'].iloc[0]
    total_change = ((current_price - first_price) / first_price) * 100
    
    print(f"Giá hiện tại: ${current_price:,.2f}")
    print(f"Cao nhất: ${highest_price:,.2f}")
    print(f"Thấp nhất: ${lowest_price:,.2f}")
    print(f"Giá trung bình: ${avg_price:,.2f}")
    print(f"Thay đổi {days} ngày: {total_change:+.2f}%")
    print(f"Khối lượng TB: {avg_volume:,.0f}")

# Test phân tích
analyze_crypto("BTCUSDT", "1day", 30)


PHÂN TÍCH BTC (30 ngày)
Đang lấy dữ liệu BTCUSDT (1day)...
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:00
Giá hiện tại: $105,306.45
Cao nhất: $111,958.32
Thấp nhất: $93,399.12
Giá trung bình: $104,949.01
Thay đổi 30 ngày: +11.37%
Khối lượng TB: 8,474
Lấy thành công 30 nến từ 2025-05-05 16:00:00 đến 2025-06-03 16:00:00
Giá hiện tại: $105,306.45
Cao nhất: $111,958.32
Thấp nhất: $93,399.12
Giá trung bình: $104,949.01
Thay đổi 30 ngày: +11.37%
Khối lượng TB: 8,474


In [49]:
# DEMO: Test hệ thống cập nhật file
print("=== DEMO: TEST HỆ THỐNG CẬP NHẬT FILE ===\n")

def demo_update_system():
    """
    Demo test hệ thống cập nhật file
    Lấy dữ liệu mới và cập nhật vào file existing
    """
    import os
    
    print("1. Lấy dữ liệu Bitcoin mới (test nhanh)...")
    test_data = get_crypto_data("BTCUSDT", "1h",30)
    
    if test_data is not None:
        print(f"   Lấy được {len(test_data)} dòng dữ liệu")
        
        print("\n2. Cập nhật vào file...")
        saved_file = save_crypto_data_smart(test_data, "BTCUSDT")
        
        if saved_file and os.path.exists(saved_file):
            print("\n3. Kiểm tra kết quả...")
            final_data = pd.read_csv(saved_file)
            print(f"   Tổng dòng trong file: {len(final_data)}")
            print(f"   Dữ liệu mới nhất: {final_data['datetime'].max()}")
            print(f"   Giá hiện tại: ${final_data['close'].iloc[-1]:,.2f}")
            
            print("\n   3 dòng cuối cùng:")
            display(final_data[['datetime', 'open', 'high', 'low', 'close', 'volume']].tail(3))
            
        print("\nDemo hoàn thành! Hệ thống đang hoạt động tốt.")
    else:
        print("Không thể lấy dữ liệu test")

print("Chạy dòng dưới để test hệ thống:")
print("demo_update_system()")

# Bỏ comment để chạy:
demo_update_system()

=== DEMO: TEST HỆ THỐNG CẬP NHẬT FILE ===

Chạy dòng dưới để test hệ thống:
demo_update_system()
1. Lấy dữ liệu Bitcoin mới (test nhanh)...
Đang lấy dữ liệu BTCUSDT (1h)...
Lấy thành công 200 nến từ 2025-05-27 18:00:00 đến 2025-06-05 01:00:00
   Lấy được 200 dòng dữ liệu

2. Cập nhật vào file...
Tìm thấy file existing: crypto_data/btc_data.csv (24 dòng)
Lỗi khi đọc file cũ: '<' not supported between instances of 'Timestamp' and 'str'
Tạo file mới...
Đã tạo file mới: crypto_data/btc_data.csv (200 dòng)

3. Kiểm tra kết quả...
   Tổng dòng trong file: 200
   Dữ liệu mới nhất: 2025-06-05 01:00:00
   Giá hiện tại: $104,796.73

   3 dòng cuối cùng:
Lấy thành công 200 nến từ 2025-05-27 18:00:00 đến 2025-06-05 01:00:00
   Lấy được 200 dòng dữ liệu

2. Cập nhật vào file...
Tìm thấy file existing: crypto_data/btc_data.csv (24 dòng)
Lỗi khi đọc file cũ: '<' not supported between instances of 'Timestamp' and 'str'
Tạo file mới...
Đã tạo file mới: crypto_data/btc_data.csv (200 dòng)

3. Kiểm tra k

,datetime,open,high,low,close,volume
197,2025-06-04 23:00:00,104823.00,104894.5,104653.66,104718.79,95.598693
198,2025-06-05 00:00:00,104718.79,104998.8,104690.00,104998.79,107.532983
199,2025-06-05 01:00:00,104998.79,105164.3,104732.00,104796.73,116.563581



Demo hoàn thành! Hệ thống đang hoạt động tốt.
